# sphinxcontrib_programoutput

In [ ]:
# Example 1: program-output runs the command at build time and shows what it printed.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.programoutput']\n")
(src / "index.rst").write_text('Output\n======\n\n.. program-output:: python3 -c "print(6 * 7)"\n')
rc = build_main(["-q", "-W", "-n", "-b", "text", str(src), str(src / "_out")])
text = (src / "_out" / "index.txt").read_text()
print("rc", rc)
print(text)
assert rc == 0
assert text.split() == ["Output", "******", "42"], text.split()

In [ ]:
# Example 2: command-output shows the command with a prompt, then its output (the template is configurable).
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.programoutput']\nprogramoutput_prompt_template = '>>> {command}\\n{output}'\n")
(src / "index.rst").write_text("Session\n=======\n\n.. command-output:: echo hello\n")
assert build_main(["-q", "-W", "-b", "text", str(src), str(src / "_out")]) == 0
text = (src / "_out" / "index.txt").read_text()
print(text)
assert ">>> echo hello" in text
assert text.strip().splitlines()[-1].strip() == "hello"

In [ ]:
# Example 3: a command that is meant to fail: `:returncode:` states the exit code the page expects.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.programoutput']\n")
(src / "index.rst").write_text(
    'Failure\n=======\n\n.. program-output:: python3 -c "import sys; print(\'bad input\'); sys.exit(2)"\n   :returncode: 2\n')
rc = build_main(["-q", "-W", "-b", "text", str(src), str(src / "_out")])
text = (src / "_out" / "index.txt").read_text()
print("rc", rc, text.split())
assert rc == 0
assert "bad input" in text

In [ ]:
# Example 4: `:shell:` runs a pipeline through the shell, `:cwd:` sets the folder it runs in, `:ellipsis:` trims lines.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
data = src / "data"
data.mkdir()
for name in ("b.txt", "a.txt", "c.txt"):
    (data / name).write_text("x")
(src / "conf.py").write_text("extensions = ['sphinxcontrib.programoutput']\n")
(src / "index.rst").write_text("Files\n=====\n\n.. program-output:: ls | sort\n   :shell:\n   :cwd: data\n\n"
                               ".. program-output:: seq 1 10\n   :ellipsis: 2\n")
assert build_main(["-q", "-W", "-b", "text", str(src), str(src / "_out")]) == 0
words = (src / "_out" / "index.txt").read_text().split()
print(words)
assert words[2:5] == ["a.txt", "b.txt", "c.txt"], words
assert words[5:] == ["1", "2", "..."], words

In [ ]:
# Example 5: strict mode fails the build when a command exits with a code the page did not expect.
import io, pathlib, tempfile
from contextlib import redirect_stderr
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinxcontrib.programoutput']\n")
(src / "index.rst").write_text('Broken\n======\n\n.. program-output:: python3 -c "raise SystemExit(3)"\n')
err = io.StringIO()
with redirect_stderr(err):
    rc = build_main(["-q", "-W", "--keep-going", "-b", "text", str(src), str(src / "_out")])
print("rc", rc, err.getvalue()[-300:])
assert rc == 1
assert "Unexpected return code 3" in err.getvalue()